In [1]:
%%time
!pip install ../input/efficientnet-pytorch/EfficientNet-PyTorch-1.0 -f ./ --no-index

ERROR: Invalid requirement: '../input/efficientnet-pytorch/EfficientNet-PyTorch-1.0': Expected package name at the start of dependency specifier
    ../input/efficientnet-pytorch/EfficientNet-PyTorch-1.0
    ^
Hint: It looks like a path. File '../input/efficientnet-pytorch/EfficientNet-PyTorch-1.0' does not exist.
CPU times: user 10.2 ms, sys: 0 ns, total: 10.2 ms
Wall time: 835 ms


In [2]:
import os
import gc
import sys
import json
import time
import cv2
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
from torchvision import models, transforms
from torch.utils.data.sampler import SequentialSampler
from efficientnet_pytorch import model as enet

KAGGLE = True
if not KAGGLE: os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
else: pass
DEVICE = torch.device('cuda')

ModuleNotFoundError: No module named 'efficientnet_pytorch'

In [3]:
TEST = True
VER = 'v0'
if KAGGLE:
    DATA_PATH = '../input/plant-pathology-2021-fgvc8'
    MDLS_PATH = f'../input/plant-models-{VER}'
else:
    DATA_PATH = './data'
    MDLS_PATH = f'./models_{VER}'
TH = .5
VOTERS = 1
TTAS = [0, 1, 2]
FOLDS = [0, 1]
IMGS_PATH = f'{DATA_PATH}/test_images' if TEST else f'{DATA_PATH}/train_images'

LABELS_ = {
    'complex': 0, 
    'frog_eye_leaf_spot': 1, 
    'healthy': 2, 
    'powdery_mildew': 3, 
    'rust': 4, 
    'scab': 5
}
LABELS = {
    0: 'complex', 
    1: 'frog_eye_leaf_spot', 
    2: 'healthy', 
    3: 'powdery_mildew', 
    4: 'rust', 
    5: 'scab'
}

start_time = time.time()

NameError: name 'KAGGLE' is not defined

In [4]:
with open(f'{MDLS_PATH}/params.json') as file:
    params = json.load(file)
print('loaded params:', params)

NameError: name 'MDLS_PATH' is not defined

In [5]:
df_sub = pd.read_csv(f'{DATA_PATH}/sample_submission.csv')
display(df_sub.head())

NameError: name 'DATA_PATH' is not defined

In [6]:
def flip(img, axis=0):
    if axis == 1:
        return img[::-1, :, ]
    elif axis == 2:
        return img[:, ::-1, ]
    elif axis == 3:
        return img[::-1, ::-1, ]
    else:
        return img

class PlantDataset(data.Dataset):
    
    def __init__(self, df, size, labels, transform=None, tta=0):
        self.df = df.reset_index(drop=True)
        self.size = size
        self.labels = labels
        self.transform = transform
        self.tta = tta
    
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        img_name = row.image
        img_path = f'{IMGS_PATH}/{img_name}'
        img = cv2.imread(img_path)
        if not np.any(img):
            print('no img file read:', img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.size, self.size))
        img = img.astype(np.float32) / 255
        if self.transform is not None:
            img = self.transform(image=img)['image']
        img = img.transpose(2, 0, 1)
        if self.labels:
            label = np.zeros(len(self.labels)).astype(np.float32)
            for lbl in row.labels.split():
                label[self.labels[lbl]] = 1.
            return torch.tensor(img), torch.tensor(label)
        else:
            img = flip(img, axis=self.tta)
            return torch.tensor(img.copy())

class EffNet(nn.Module):
    
    def __init__(self, params, out_dim):
        super(EffNet, self).__init__()
        self.enet = enet.EfficientNet.from_name(params['backbone'])
        nc = self.enet._fc.in_features
        self.enet._fc = nn.Identity()
        self.myfc = nn.Sequential(
            nn.Dropout(params['dropout']),
            nn.Linear(nc, int(nc / 4)),
            nn.ELU(),
            #nn.BatchNorm1d(int(nc / 4)), 
            nn.Dropout(params['dropout']),
            nn.Linear(int(nc / 4), out_dim)
        )
        
    def extract(self, x):
        return self.enet(x)
    
    def forward(self, x):
        x = self.extract(x)
        x = self.myfc(x)
        return x

In [7]:
models = []
params['backbone'] = 'efficientnet-b1'
for n_fold in FOLDS:
    model = EffNet(params, out_dim=len(LABELS_))
    path = '{}/model_best_{}.pth'.format(MDLS_PATH, n_fold)
    state_dict = torch.load(path, map_location=torch.device('cpu'))
    model.load_state_dict(state_dict)
    model.float()
    model.eval()
    model.cuda()
    models.append(model)
    print('loaded:', path)
del state_dict, model
gc.collect();

NameError: name 'params' is not defined

In [8]:
datasets, loaders = [], []
for tta in TTAS:
    dataset = PlantDataset(
        df=df_sub,
        size=params['img_size'],
        labels=None,
        transform=None,
        tta=tta)
    datasets.append(dataset)
    loader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=8, 
        sampler=SequentialSampler(dataset), 
        num_workers=2)
    loaders.append(loader)

NameError: name 'TTAS' is not defined

In [9]:
def get_labels(row, labels, th):
    row = [i for i, e in enumerate(row) if e > th]
    row = [labels[i] for i in row]
    row = ' '.join(row) if row else 'healthy'
    return row

logits = []
with torch.no_grad():
    for i, model in enumerate(models):
        for j, loader in enumerate(loaders):
            logits_tta = []
            for img_data in loader:
                img_data = img_data.to(DEVICE)
                preds = np.squeeze(model(img_data).sigmoid().cpu().numpy())
                logits_tta.append(preds)
            print('model {} | loader {} -> done'.format(i, j))
            logits.append(logits_tta)
all_preds = np.squeeze(np.array(np.mean(logits, axis=0)))
df_sub['labels'] = [get_labels(x, LABELS, TH) for x in list(all_preds)]

elapsed_time = time.time() - start_time
print(f'time elapsed: {elapsed_time // 60:.0f} min {elapsed_time % 60:.0f} sec')

/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


TypeError: iteration over a 0-d array

In [10]:
print('value counts:')
print(df_sub.labels.value_counts())
df_sub.head()

value counts:


NameError: name 'df_sub' is not defined

In [11]:
df_sub.to_csv('submission.csv', index=False)

NameError: name 'df_sub' is not defined